In [18]:
import json
import boto3
import pandas as pd

session = boto3.Session(profile_name="broker-secrets")
client = session.client("secretsmanager", region_name="ap-south-1")

response = client.get_secret_value(SecretId="/trading/brokers/mastertrust/vaibhav")
secret = json.loads(response["SecretString"])

for key, value in secret.items():
    print(f"{key}: {value}")


client_id: 090009421_U
password: He11o-W0rld
redirect_url: https://tarantularesearch.com/
secret_key: EeGrgnozNQ2yZEU1G0wWX5hFw29ZcTFG15TA4vNnpiLhI97RlUfW69BD5wD4Inq1
access_token: cccf016de153bb2520cb77fdd5d82a31e861b1a34412b3594c8de760fe01b3ca
refresh_token: 0df3df8a7d4ffbd6cad23d43fdfb20b8f8f1b2bdbd4baff2f6a8f96c103ba94b


In [19]:
import json
import requests

base_url = "https://midlive.mastertrust.co.in/NorenWClientAPI/"
uid = secret["client_id"].split("_")[0]
access_token = secret["access_token"]


def call(endpoint, body=None):
    payload = {"uid": uid, "actid": uid, **(body or {})}
    data = f"jData={json.dumps(payload)}"
    headers = {"Authorization": f"Bearer {access_token}"}

    response = requests.post(base_url + endpoint, data=data, headers=headers)
    response.raise_for_status()
    return response.json()


result = call("UserDetails")
result


{'request_time': '13:24:22 21-08-2026',
 'uname': 'PRIYA SETHI',
 'm_num': '9873000826',
 'email': 'vsethi@quantxindia.com',
 'access_type': ['TT',
  'WEB',
  'WEBE',
  'WEBS',
  'MOB',
  'MIM',
  'MIMEXE',
  'MIMWEB',
  'API',
  'API2',
  'MIMMOB',
  'MIMAPI'],
 'exarr': ['BCD', 'BFO', 'BSE', 'CDS', 'MCX', 'NCOM', 'NCX', 'NFO', 'NSE'],
 'prarr': [{'prd': 'C',
   's_prdt_ali': 'CNC',
   'exch': ['NSE', 'BSE', 'NSE', 'BSE']},
  {'prd': 'M',
   's_prdt_ali': 'NRML',
   'exch': ['NFO', 'BFO', 'CDS', 'BCD', 'MCX', 'NSE', 'BSE', 'NCOM', 'NCX']},
  {'prd': 'I',
   's_prdt_ali': 'MIS',
   'exch': ['NSE', 'BSE', 'NFO', 'BFO', 'CDS', 'BCD', 'MCX', 'NCOM']},
  {'prd': 'H',
   's_prdt_ali': 'CO',
   'exch': ['NSE', 'BSE', 'NFO', 'BFO', 'CDS', 'BCD', 'MCX', 'NCOM']},
  {'prd': 'B',
   's_prdt_ali': 'BO',
   'exch': ['NSE', 'BSE', 'NFO', 'BFO', 'CDS', 'BCD', 'MCX', 'NCOM']},
  {'prd': 'F', 's_prdt_ali': 'MTF', 'exch': ['NSE', 'BSE']},
  {'prd': 'P', 's_prdt_ali': 'MTFS', 'exch': ['NSE', 'BSE']}],
 

In [20]:
result = call("PositionBook")
result

{'stat': 'Not_Ok',
 'request_time': '13:24:23 21-08-2026',
 'emsg': 'Error Occurred : 5 "no data"'}

In [21]:
order = call("PlaceOrder", {
    "exch": "NSE",
    "tsym": "RELIANCE-EQ",
    "qty": "1",
    "prc": "0",
    "prd": "I",        # I = Intraday
    "trantype": "B",   # B = Buy, S = Sell
    "prctyp": "MKT",   # Market order, so prc must be "0"
    "ret": "DAY",
})
order

{'request_time': '13:24:24 21-08-2026',
 'stat': 'Not_Ok',
 'emsg': 'Rejected : ALGO_CHK: MKT Order type not allowed for API order'}

In [22]:
call("SearchScrip", {"stext": "RELIANCE", "exch": "NSE"})

{'stat': 'Ok',
 'values': [{'exch': 'NSE',
   'token': '2885',
   'tsym': 'RELIANCE-EQ',
   'cname': 'RELIANCE INDUSTRIES LTD',
   'instname': 'EQ',
   'symname': 'RELIANCE',
   'seg': 'EQT',
   'pp': '2',
   'ls': '1',
   'ti': '0.10'}]}

In [23]:
# 1. Get scrip details including tick size
scrip = call("SearchScrip", {"stext": "RELIANCE-EQ", "exch": "NSE"})["values"][0]
token = scrip["token"]
tick_size = float(scrip["ti"])  # e.g. 0.10

# 2. Get current LTP
quote = call("GetQuotes", {"exch": "NSE", "token": token})
ltp = float(quote["lp"])

# 3. Compute a marketable buy price, rounded to the nearest valid tick
raw_price = ltp * 1.002
buy_price = round(raw_price / tick_size) * tick_size
buy_price = round(buy_price, 2)  # clean up float precision

order = call("PlaceOrder", {
    "exch": "NSE",
    "tsym": "RELIANCE-EQ",
    "qty": "1",
    "prc": str(buy_price),
    "prd": "I",
    "trantype": "B",
    "prctyp": "LMT",
    "ret": "DAY",
})
order

{'request_time': '13:24:29 21-08-2026',
 'stat': 'Ok',
 'norenordno': '26082100072543'}

In [24]:
pd.DataFrame(call("OrderBook"))


,stat,norenordno,kidid,uid,actid,exch,tsym,rejby,src_uid,qty,...,brnchid,C,s_prdt_ali,prd,status,st_intrn,norentm,algo_id,rqty,ui_dev_code
0,Ok,26082100072543,1,090009421,090009421,NSE,RELIANCE-EQ,RED,,1,...,BDL,C,MIS,I,REJECTED,REJECTED,13:24:29 21-08-2026,99999,0,090009421_U
1,Ok,26082100072125,1,090009421,090009421,NSE,RELIANCE-EQ,RED,,1,...,BDL,C,MIS,I,REJECTED,REJECTED,13:21:41 21-08-2026,99999,0,090009421_U
2,Ok,26082100066338,1,090009421,090009421,NSE,RELIANCE-EQ,RED,,1,...,BDL,C,MIS,I,REJECTED,REJECTED,12:47:08 21-08-2026,99999,0,090009421_U
3,Ok,26082100066201,1,090009421,090009421,NSE,RELIANCE-EQ,ORA,,1,...,BDL,C,MIS,I,REJECTED,REJECTED,12:45:58 21-08-2026,99999,NaN,090009421_U
